# Act 1 · The Bipartite Identifier Graph, Promiscuous Hubs & IDF Rarity

### Why Naive Graph Resolution Fails in Retail & Consumer CDPs
In retail, loyalty, and contact-centre data, **not all shared identifiers mean two records are the same person**:
- **Promiscuous Hubs (`degree > 25`):** Store checkout kiosks (`EM:kiosk.mel@store-checkout.com.au`, `DV:KIOSK-TERMINAL-AU-01`) and contact-centre fallback numbers (`PH:+61390002222`) appear on **hundreds of unrelated customer profiles**.
- **Shared Household Identifiers (`degree 2..5`):** Family email addresses (`EM:thegills@gmail.com`) and home landlines are legitimately shared by **2 to 4 distinct family members** living at the same address.
- **Temporal Identifier Supersession (`has_identifier_history`):** Mobile numbers and emails are recycled or updated over time (`valid_from`, `valid_to`, `is_current`).

In Stage 40 (`40_block.sql`, sections `40e`–`40h`), we construct a **Bipartite Profile $\leftrightarrow$ Identifier Graph** directly in BigQuery SQL and compute each identifier's **Inverse Document Frequency (`idf_weight = LN(N / degree)`)**.

In [1]:
%load_ext bigquery_magics

import bigquery_magics
import pandas as pd
from IPython.display import display
from pathlib import Path

# ---------------------------------------------------------------------------
# Set PROJECT if running outside a checkout that has demo/config.env
PROJECT = ""          # e.g. "all-things-cdp"
DATASET = ""          # leave blank for the default, "cdp"
# ---------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 52)

CFG, CONFIG, _source = {}, None, None

def _find_config():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (base / "config.env", base / "demo" / "config.env"):
            if candidate.is_file():
                return candidate
    return None

if not PROJECT:
    CONFIG = _find_config()
    if CONFIG:
        for _line in CONFIG.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                CFG[_k.strip()] = _v.split("#", 1)[0].strip().strip('"').strip("'")
        PROJECT = CFG.get("CDP_PROJECT", "")
        _source = str(CONFIG)

if not PROJECT:
    import os
    PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT") or "all-things-cdp"
    _source = "environment / default"

DATASET = DATASET or CFG.get("CDP_DS") or CFG.get("CDP_DATASET_PREFIX") or "cdp"
LOCATION = CFG.get("CDP_LOCATION", "US")

bigquery_magics.context.project = PROJECT
bigquery_magics.context.location = LOCATION
bigquery_magics.context.progress_bar_type = None

print(f"project    {PROJECT}   (from {_source})")
print(f"dataset    {DATASET}   (location {LOCATION})")

project    all-things-cdp   (from demo/config.env)
dataset    cdp   (location US)

## 1 · Identifier Degree & IDF Rarity Distribution (`cdp.identifier`)

Let's inspect `cdp.identifier` by identifier type (`email`, `phone`, `loyalty_account`, `device_id`). Notice the sharp separation between:
- **Unique personal identifiers (`degree = 1`)** and **small person/household clusters (`degree 2..5`)**, where `idf_weight` is high (`8.5` to `9.9`).
- **Promiscuous hubs (`degree > 25`)**, where `degree` jumps to **`419..575`** and `idf_weight` drops to **`3.55..3.87`**.

In [2]:
%%bigquery df_rarity
SELECT
  identifier_type,
  COUNT(*) AS distinct_identifiers,
  COUNTIF(degree = 1) AS degree_1_unique,
  COUNTIF(degree BETWEEN 2 AND 5) AS degree_2_to_5_household_or_person,
  COUNTIF(degree BETWEEN 6 AND 25) AS degree_6_to_25,
  COUNTIF(is_promiscuous) AS promiscuous_hubs,
  MAX(degree) AS max_degree,
  ROUND(AVG(idf_weight), 2) AS avg_idf_weight
FROM `cdp.identifier`
GROUP BY identifier_type
ORDER BY distinct_identifiers DESC

identifier_type,distinct_identifiers,degree_1_unique,degree_2_to_5_household_or_person,degree_6_to_25,promiscuous_hubs,max_degree,avg_idf_weight
email,7742,5208,2531,0,3,575,9.65
phone,6672,4426,2243,0,3,522,9.66
loyalty_account,4309,3312,997,0,0,4,9.73
device_id,1,0,0,0,1,547,3.6


## 2 · Inspecting the 7 Promiscuous Hubs vs Shared Household Identifiers

Below are the top high-degree identifiers in `cdp.identifier`.
- The top **7 rows** (`is_promiscuous = TRUE`) are store kiosks and service-desk defaults spanning all 7 source systems (`CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT`).
- If even a single one of these hubs is allowed to transitively link profiles, hundreds of strangers collapse into one cluster — and because profiles touch multiple hubs, **all 7 hubs bridge together into a single 5,195-profile mega-cluster ("the hairball")**.
- Below the 7 hubs, the next highest-degree identifiers (`degree = 4`, `idf_weight = 8.517`) are **shared household emails** (`EM:thegills@gmail.com`, `EM:themorriss@bigpond.com`, `EM:thekramers@gmail.com`).

In [3]:
%%bigquery df_hubs
SELECT
  identifier_id,
  identifier_type,
  degree,
  degree_all_time,
  ROUND(idf_weight, 3) AS idf_weight,
  is_promiscuous,
  sources
FROM `cdp.identifier`
ORDER BY degree DESC, identifier_id
LIMIT 12

identifier_id,identifier_type,degree,degree_all_time,idf_weight,is_promiscuous,sources
EM:service.desk@contact-centre.com.au,email,575,575,3.549,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
EM:kiosk.mel@store-checkout.com.au,email,559,559,3.577,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
DV:KIOSK-TERMINAL-AU-01,device_id,547,547,3.599,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
EM:kiosk.syd@store-checkout.com.au,email,529,529,3.632,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
PH:+61390002222,phone,522,522,3.646,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
PH:+61290001111,phone,516,516,3.657,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
PH:+61890003333,phone,419,419,3.866,True,CALL|CRM|ECOM|ENRICH|LOYALTY|POS|SUPPORT
AC:ACC-2827885,loyalty_account,4,4,8.517,False,LOYALTY|POS
AC:ACC-4980450,loyalty_account,4,4,8.517,False,LOYALTY|POS
AC:ACC-6720578,loyalty_account,4,4,8.517,False,LOYALTY|POS|SUPPORT


## 3 · Querying the Native BigQuery Property Graph (`cdp.cdp_identity_graph`) with ISO GQL

Stage 70 (`70_graph.sql`) declares `cdp.cdp_identity_graph` as a native BigQuery `PROPERTY GRAPH` over both the **pre-resolution Bipartite Identifier Graph** (`SourceRecord` $\xrightarrow{\text{HAS\_IDENTIFIER}}$ `Identifier`) and the **post-resolution Entity Graph** (`SourceRecord` $\xrightarrow{\text{RESOLVES\_TO}}$ `Person` $\xrightarrow{\text{MEMBER\_OF}}$ `Household`).

We can traverse `cdp.cdp_identity_graph` directly using ISO/IEC 39075 **GQL (`GRAPH_TABLE`)** to inspect records attached to a promiscuous kiosk hub versus a household email.

In [4]:
%%bigquery df_gql_hub
SELECT
  identifier_id,
  identifier_type,
  degree,
  is_promiscuous,
  COUNT(record_id) AS attached_profiles,
  STRING_AGG( raw_name, ', ' ORDER BY record_id LIMIT 5 ) AS sample_names
FROM GRAPH_TABLE(
  `cdp.cdp_identity_graph`
  MATCH (r:SourceRecord)-[h:HAS_IDENTIFIER]->(i:Identifier)
  WHERE i.identifier_id IN ('EM:kiosk.mel@store-checkout.com.au', 'EM:thegills@gmail.com')
  COLUMNS (
    i.identifier_id AS identifier_id,
    i.identifier_type AS identifier_type,
    i.degree AS degree,
    i.is_promiscuous AS is_promiscuous,
    r.record_id AS record_id,
    r.raw_name AS raw_name
  )
)
GROUP BY identifier_id, identifier_type, degree, is_promiscuous
ORDER BY degree DESC

identifier_id,identifier_type,degree,is_promiscuous,attached_profiles,sample_names
EM:kiosk.mel@store-checkout.com.au,email,559,True,559,"Michael Harding, Kanokwan Prasertsak, Sutton Daskalakis, Lap Trinh, Caden Avramidis"
EM:thegills@gmail.com,email,4,False,4,"Tanmay Gill, Shaanthi Gill, Shanthi Gill, Tanmay Gill"


## 4 · Why Weighted Connected Components Alone Is Not Enough

Look at the second row above (`EM:thegills@gmail.com`, `degree = 4`, `idf_weight = 8.517`):
- It is **not** a promiscuous hub (`degree = 4 <= 25`), and its IDF weight (`8.517`) is **above** the `weighted_cc` threshold (`7.0`).
- Yet the 4 profiles attached to `EM:thegills@gmail.com` belong to **three different family members** (`Harpreet Gill`, `Jaspreet Gill`, `Gurpreet Gill`) sharing a family email!
- Pure IDF-weighted graph projection (`weighted_cc`) blindly merges all 4 records into a single person (`100` household false positives).
- In **Act 2 (`03_hybrid_search_and_2hop_features.ipynb`)**, we see how **2-Hop Neighbourhood Features + Hybrid Search** separate household members and recycled identifiers without training a PyTorch GNN.